# Diagnóstico de Calidad de Datos - EV06
**Proyecto:** Optimización del Mantenimiento Predictivo y Consumo Energético en Flotas de Vehículos Eléctricos e Híbridos.
**Autor:** Sebastian Lein Castro Grajales

## Momento 1. Reconocimiento del proyecto y de los datasets

**1. Propósito del proyecto:** 
Construir modelos predictivos para identificar riesgos de degradación de baterías (SoH) y analizar la eficiencia energética de estaciones de carga frente a la topografía y clima local.

**2. Pregunta analítica:** 
¿Cuáles son los patrones de degradación de las celdas de litio bajo cargas rápidas y cómo se distribuye la demanda energética real en la infraestructura de carga urbana?

**3. Datasets seleccionados:**
- `ev_battery_kaggle.csv`: Telemetría de baterías, temperatura, ciclos de carga y Estado de Salud (SoH).
- `estaciones_carga_colombia.csv`: Registros de consumo energético por municipio y operador.

**4. Aspectos de calidad a revisar:**
Se requiere evaluar la completitud de las variables (valores nulos en sensores), la coherencia de los tipos de datos numéricos (como los porcentajes de SoH) y la validez de los rangos lógicos (ej. evitar temperaturas extremas irreales o ciclos negativos).

In [ ]:
import pandas as pd

# Carga de los datasets
df_bat = pd.read_csv('../data/raw/ev_battery_kaggle.csv', sep=';')
df_est = pd.read_csv('../data/raw/estaciones_carga_colombia.csv', sep=';')

print("Carga exitosa:")
print(f"- Dataset Baterías: {df_bat.shape[0]} filas, {df_bat.shape[1]} columnas.")
print(f"- Dataset Estaciones: {df_est.shape[0]} filas, {df_est.shape[1]} columnas.")

In [ ]:
print("--- 1. HEAD Y TAIL (Baterías) ---")
display(df_bat.head(3))
display(df_bat.tail(3))

print("\n--- 2. INFO (Ambos Datasets) ---")
df_bat.info()
print("-" * 30)
df_est.info()

print("\n--- 3. DESCRIBE NUMÉRICO Y CATEGÓRICO (Estaciones) ---")
display(df_est.describe())
display(df_est.describe(include='object'))

## Momento 3. Inspección de la estructura de los datasets (Diagnóstico)

| Dataset | Filas | Columnas | Tipos de datos | Observaciones |
| :--- | :--- | :--- | :--- | :--- |
| **Baterías EV** | 1000 | 5 | 3 object, 1 int64, 1 float64 | La variable `soh_porcentaje` se está leyendo como texto (object) en lugar de numérico decimal debido a que contiene el símbolo `%`. Hay valores nulos en `temp_max_celda_C` (16 nulos). |
| **Estaciones Carga** | 500 | 5 | 4 object, 1 int64 | `kwh_entregados_mes` se cargó como `object` (texto), lo que impide cálculos matemáticos. `operador` presenta 51 valores nulos. `fecha_reporte` es tipo texto, no formato *datetime*. |



## Momento 4. Diagnóstico inicial de calidad

Tras la exploración, se identificaron múltiples anomalías que deben tratarse antes del modelado predictivo:

| Dataset | Problema identificado | Evidencia encontrada | Posible impacto | Acción propuesta |
| :--- | :--- | :--- | :--- | :--- |
| **Baterías EV** | Valores faltantes | 16 registros nulos en `temp_max_celda_C`. | Falla en algoritmos de Machine Learning que no soportan NaNs. | Imputar con la mediana térmica o eliminar las filas afectadas. |
| **Baterías EV** | Tipo de dato erróneo | `soh_porcentaje` es de tipo `object` por el símbolo `%`. | Imposibilidad de calcular estadísticas o correlaciones matemáticas del estado de salud. | Remover el carácter `%` y castear la columna a `float64`. |
| **Baterías EV** | Inconsistencia lógica | Registro con `ciclos_carga` = -500. | Ruido estadístico; físicamente imposible tener ciclos negativos. | Aplicar valor absoluto (500) o eliminar la fila. |
| **Estaciones** | Valores faltantes | 51 nulos en la columna categórica `operador`. | Cálculos de cuota de mercado o distribución energética por empresa inexactos. | Rellenar con la etiqueta `"No especificado"`. |
| **Estaciones** | Texto en numéricos | `kwh_entregados_mes` contiene el texto `"Sin lectura"`. | Toda la columna se volvió `object`, bloqueando sumas de consumo energético. | Reemplazar `"Sin lectura"` por `NaN` y convertir a `float64`. |
| **Estaciones** | Error tipográfico | El municipio `"Medellín"` está escrito como `"Medelín"` en un registro. | Al hacer un `.groupby()`, Pandas creará una ciudad fantasma aparte, dividiendo los datos. | Estandarizar textos con `.replace('Medelín', 'Medellín')`. |